# DART + CRNN frente a V6 en personas de Sleep-Cassette

Pregunta: tras entrenar con las mismas 84 noches de desarrollo, ¿qué sistema obtiene mayor F1 macro sobre los mismos segmentos de personas de Cassette? Se compara la mezcla DART + CRNN con la fase C de [model_v6.ipynb](model_v6.ipynb), que usa CRNN, LightGBM contextual y decodificación Viterbi. La configuración de ambos sistemas queda fijada antes de evaluar.

Es una comparación retrospectiva dentro de Sleep-EDF. Las personas disponibles ya participaron en el desarrollo histórico del proyecto. El modelo mixto se vuelve a entrenar desde cero. Para reproducir la corrida, usa el entorno Python del proyecto y los datos y modelos de la revisión del repositorio (Git y DVC).

**Arquitecturas.** DART y la CRNN espectral producen dos distribuciones de probabilidad que se combinan al final. En V6 fase C, las convoluciones separadas para EEG, EOG y EMG, la atenci?n y una LSTM bidireccional generan un vector de 64 componentes por ?poca. LightGBM clasifica tres vectores vecinos (192 componentes) y el HMM decodifica la secuencia completa de la noche.

En este sorteo, las diez personas de prueba habían estado en el entrenamiento del experimento mixto anterior. Se descartan aquellos pesos y se vuelven a ajustar ambas ramas exclusivamente con las 84 noches acordadas. La elección histórica de arquitectura puede seguir influyendo en la comparación.

In [1]:
import hashlib
import json
import os
import sys
from pathlib import Path

os.environ.setdefault("TF_ENABLE_ONEDNN_OPTS", "0")
os.environ.setdefault("TF_NUM_INTRAOP_THREADS", "5")
os.environ.setdefault("TF_NUM_INTEROP_THREADS", "1")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd
import tensorflow as tf
from lightgbm import Booster
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix, f1_score

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / ".dvc/config").is_file())
DATA = ROOT / "datos"
AUX = ROOT / "Notebooks/experiments/auxiliares_dart_crnn"
sys.path[:0] = [str(ROOT / "packages/sleep-staging/src"), str(ROOT), str(AUX)]
from crnn_contextual_mixto import entrenar_crnn
from scripts.compare_cassette_hybrid import contextual_embeddings, decode_night, hybrid_inputs
from sleep_staging import PreprocessingConfig, PreprocessingPipeline
from sleep_staging.datasets import discover_sleep_cassette_records, discover_sleep_telemetry_records
from sleep_staging.inference.spectral import crnn_probabilities, spectral_input
from sleep_staging.training import create_lightgbm_classifier, load_or_build_supervised_dataset

def leer_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

experimento = DATA / "experimentos/dart_crnn_mixto/dart_666_crnn_27_20261119_191036"
ref = DATA / "referencias/hibrido_v6"
spec = leer_json(DATA / "experimentos/lightgbm_st_sc/experimento.json")
hiper = leer_json(experimento / "hiperparametros.json")
fusion = leer_json(experimento / "fusion.json")
receta = leer_json(experimento / "referencia_seleccion_80_20.json")["contrato"]
hmm = leer_json(ref / "hmm.json")
clases = np.array(spec["classes"])
clases_v6 = np.array(hmm["classes"])
assert set(clases) == set(clases_v6) and clases_v6[0] == "W"
assert (hiper["iteraciones_dart"], hiper["epocas_crnn"], fusion["porcentaje_dart"]) == (666, 27, 59)
tf.config.experimental.enable_op_determinism()

## Partición por persona

V6 ordena los registros por nombre. Usa todas las noches de Telemetry y las primeras 40 de Cassette para desarrollo. Dentro hace cinco pliegues por sujeto y en cada ronda aparta el 15 % de los sujetos restantes para detener la CRNN según la pérdida de validación. LightGBM compara 30 configuraciones en Optuna por F1 macro con cinco pliegues por sujeto. Para ese paso se usan los vectores del extractor maestro ajustado en todo desarrollo, por lo que el F1 interno de Optuna depende de esa representación. Las siguientes 20 noches de Cassette forman su prueba publicada. La tabla comprueba el número real de personas y el cruce en el límite.

Para el test compartido se sortean diez personas con semilla 42 y se incluyen todas sus noches. Se excluyen los sujetos de desarrollo y del holdout publicado de V6, los de la selección previa de iteraciones, épocas y mezcla, y los del experimento tabular que aportó los parámetros base. El sorteo usa solo identificadores.

In [2]:
telemetry = discover_sleep_telemetry_records(DATA / "sleep-telemetry")
cassette = discover_sleep_cassette_records(DATA / "sleep-cassette")
desarrollo = telemetry + cassette[:40]
publicado = cassette[40:60]

def sujetos(registros):
    return {r.subject_id for r in registros}

assert [r.record_id for r in desarrollo] == hmm["training_records"]
assert len(desarrollo) == 84 and len(sujetos(desarrollo)) == 43
excluidos = (sujetos(desarrollo + publicado) | set(hiper["personas_prueba"])
             | set(fusion["personas_prueba"]) | set(spec["train_subjects"])
             | set(spec["validation_subjects"]))
candidatos = sorted(sujetos(cassette) - excluidos)
elegidos = set(np.random.default_rng(42).choice(candidatos, 10, replace=False))
prueba = [r for r in cassette if r.subject_id in elegidos]
assert len(sujetos(prueba)) == 10 and sujetos(prueba).isdisjoint(excluidos)
for nombre, grupo in [("Desarrollo común", desarrollo), ("Prueba publicada de V6", publicado),
                      ("Prueba compartida", prueba)]:
    print(f"{nombre}: {len(sujetos(grupo))} personas, {len(grupo)} noches")
print("Cruce de sujetos en el protocolo publicado:", sorted(sujetos(desarrollo) & sujetos(publicado)))
print("Personas del test compartido:", ", ".join(sorted(elegidos)))
print("Noches del test compartido:", ", ".join(r.record_id for r in prueba))

Desarrollo común: 43 personas, 84 noches
Prueba publicada de V6: 11 personas, 20 noches
Prueba compartida: 10 personas, 19 noches
Cruce de sujetos en el protocolo publicado: ['SC420']
Personas del test compartido: SC433, SC436, SC442, SC449, SC450, SC456, SC459, SC462, SC465, SC475
Noches del test compartido: SC4331, SC4332, SC4362, SC4421, SC4422, SC4491, SC4492, SC4501, SC4502, SC4561, SC4562, SC4591, SC4592, SC4621, SC4622, SC4651, SC4652, SC4751, SC4752


## Preparación de las 84 noches de desarrollo

Ambos métodos reciben las mismas etiquetas de entrenamiento. V6 retiene las etapas anotadas entre el primer y el último segmento de sueño con 30 minutos de margen a cada lado. Se aplica ese criterio a las filas tabulares y a las etiquetas que contribuyen a la pérdida de la CRNN. Cada arquitectura conserva su propio preprocesamiento de señal y su contexto temporal por noche. La clave de cada observación es la pareja de registro e índice original de época de 30 segundos.

In [3]:
def tabla_recortada(registros, cohorte):
    perfil = spec["preprocessing_" + cohorte]
    pipeline = PreprocessingPipeline(PreprocessingConfig(**perfil))
    ds = load_or_build_supervised_dataset(registros, pipeline, DATA / "processed/supervised", verbose=False)
    filas = ds.metadata[["record_id", "subject_id", "epoch_index"]].assign(etapa=ds.labels)
    dormidas = filas.loc[filas.etapa.ne("W")].groupby("record_id").epoch_index.agg(["min", "max"])
    inicio = filas.record_id.map(dormidas["min"]) - 60
    fin = filas.record_id.map(dormidas["max"]) + 60
    conservar = inicio.isna() | filas.epoch_index.between(inicio, fin)
    x = ds.features.loc[conservar].reindex(columns=spec["feature_columns"]).astype("float32")
    return x.reset_index(drop=True), filas.loc[conservar].reset_index(drop=True)

def entrada_espectral(registro):
    with registro.psg_path.open("rb") as archivo:
        huella = hashlib.sha256()
        for bloque in iter(lambda: archivo.read(2**20), b""):
            huella.update(bloque)
    carpeta = DATA / "processed/crnn_espectral" / hiper["receta_espectral"]
    ruta = carpeta / f"{registro.psg_path.stem}-{huella.hexdigest()}.npy"
    if ruta.exists():
        return np.load(ruta, mmap_mode="r", allow_pickle=False)
    return spectral_input(registro.psg_path, receta["canales_crnn"], receta["bandas_hz"])

partes = [tabla_recortada(telemetry, "telemetry")]
partes.extend(tabla_recortada([r], "cassette") for r in cassette[:40])
X_train = pd.concat([x for x, _ in partes], ignore_index=True)
train = pd.concat([m for _, m in partes], ignore_index=True)
del partes
assert len(train) == len(X_train) == hmm["training_epochs"]
assert set(train.record_id) == set(hmm["training_records"])
assert set(train.subject_id).isdisjoint(elegidos)
assert not train.duplicated(["record_id", "epoch_index"]).any()
entradas = {r.record_id: entrada_espectral(r) for r in desarrollo}
print(f"{len(train):,} segmentos de entrenamiento, {X_train.shape[1]:,} variables tabulares.")

85,796 segmentos de entrenamiento, 2,280 variables tabulares.


## Ajuste de DART y CRNN

Se entrena una sola configuración. La prueba compartida se abre después del ajuste. Las probabilidades se combinan mediante la misma media geométrica de la receta original, con 59 % para DART y 41 % para la CRNN.

In [4]:
parametros = dict(spec["parameters"], n_estimators=hiper["iteraciones_dart"], n_jobs=6)
dart = create_lightgbm_classifier(**parametros)

def mostrar_progreso(env):
    if (env.iteration + 1) % 100 == 0:
        print(f"DART · {env.iteration + 1} iteraciones", flush=True)

dart.fit(X_train, train.etapa, callbacks=[mostrar_progreso])
np.testing.assert_array_equal(dart.classes_, clases)
del X_train
red = entrenar_crnn(train, entradas, clases=clases, desfases=receta["desfases"],
                    semilla=42, epocas=hiper["epocas_crnn"])
del entradas

DART · 100 iteraciones


DART · 200 iteraciones


DART · 300 iteraciones


DART · 400 iteraciones


DART · 500 iteraciones


DART · 600 iteraciones


... (2 outputs left out)

CRNN · 15/27 épocas


CRNN · 20/27 épocas


CRNN · 25/27 épocas


CRNN · 27/27 épocas


## Comparación pareada

Se carga el V6 de la fase C con su HMM entrenado en las 84 noches. Cada modelo conserva sus propias entradas. Para puntuar, se comprueban el índice original y la etapa real de cada segmento de cada noche antes de unir las dos predicciones.

In [5]:
# Huellas de los tres archivos distribuidos con DVC. La red se reempaquetó
# para compatibilidad de Keras conservando los pesos, según agent.md.
artefactos_v6 = {
    "modelo_crnn_maestro_optimo_v7.keras": "6256745bd28144dd4280e8641f4a19e25166a20b2c30e20e991f117eafd4f0bb",
    "contextual_192d.txt": "042e08cd66e9ca4aca35f243efe4d448b8319f5c7a22301dd7369d01b964beb0",
    "hmm.json": "7ad21fa07b6d33b25d9e0065167da38a531de508b6716f1dacbe5cb72d378cd9",
}
for nombre, huella in artefactos_v6.items():
    assert hashlib.sha256((ref / nombre).read_bytes()).hexdigest() == huella, nombre

v6 = tf.keras.models.load_model(ref / "modelo_crnn_maestro_optimo_v7.keras", compile=False)
extractor = tf.keras.Model(v6.inputs, v6.get_layer("embeddings_latentes_64d").output)
cabezal = Booster(model_file=str(ref / "contextual_192d.txt"))
A, pi = np.asarray(hmm["A"]), np.asarray(hmm["pi"])

@tf.function(reduce_retracing=True)
def extraer_64d(entradas, training=False):
    return extractor(entradas, training=training)

def evaluar_v6(r):
    idx, real, senal = hybrid_inputs(r, receta["canales_crnn"], clases_v6)
    representaciones = contextual_embeddings(extraer_64d, senal)
    p = cabezal.predict(representaciones, num_threads=4)
    assert p.shape == (len(idx), len(clases_v6)) and np.isfinite(p).all()
    return pd.DataFrame({"record_id": r.record_id, "epoch_index": idx, "etapa_v6": real,
                         "v6": clases_v6[decode_night(p, A, pi)]})

noches_v6 = []
for i, r in enumerate(prueba, 1):
    noches_v6.append(evaluar_v6(r))
    print(f"V6 · {i}/{len(prueba)} · {r.record_id}", flush=True)
pred_v6 = pd.concat(noches_v6, ignore_index=True)

V6 · 1/19 · SC4331


V6 · 2/19 · SC4332


V6 · 3/19 · SC4362


V6 · 4/19 · SC4421


V6 · 5/19 · SC4422


V6 · 6/19 · SC4491


... (9 outputs left out)

V6 · 16/19 · SC4651


V6 · 17/19 · SC4652


V6 · 18/19 · SC4751


V6 · 19/19 · SC4752


In [6]:
peso = fusion["porcentaje_dart"] / 100

def evaluar_mixto(r):
    x, filas = tabla_recortada([r], "cassette")
    p_dart = dart.booster_.predict(x, num_threads=4)
    p_crnn = crnn_probabilities(red, entrada_espectral(r), receta["desfases"], len(clases))
    p_crnn = p_crnn[filas.epoch_index.to_numpy()]
    assert p_dart.shape == p_crnn.shape == (len(filas), len(clases))
    log_p = peso * np.log(np.maximum(p_dart, 1e-7)) + (1 - peso) * np.log(np.maximum(p_crnn, 1e-7))
    assert np.isfinite(log_p).all()
    return filas.assign(mixto=clases[log_p.argmax(axis=1)])

pred_mixto = pd.concat([evaluar_mixto(r) for r in prueba], ignore_index=True)
claves = ["record_id", "epoch_index"]
resultados = pred_mixto.merge(pred_v6, on=claves, how="outer", validate="one_to_one", indicator=True)
assert resultados._merge.eq("both").all()
np.testing.assert_array_equal(resultados.etapa, resultados.etapa_v6)
assert set(resultados.subject_id) == elegidos and set(resultados.record_id) == {r.record_id for r in prueba}

## F1 macro y diferencia entre los dos modelos

El resultado principal es el F1 macro de las cinco etapas sobre todos los segmentos comunes. El intervalo percentil del 95 % se calcula con 5.000 remuestras pareadas de personas. Cada persona aporta todas sus noches a ambos modelos en cada remuestra. El intervalo describe la incertidumbre asociada a estas diez personas con los modelos ya ajustados.

In [7]:
modelos = {"mixto": "DART + CRNN", "v6": "V6 fase C"}
resumen = []
for columna, nombre in modelos.items():
    reales, pred = resultados.etapa, resultados[columna]
    por_clase = f1_score(reales, pred, labels=clases_v6, average=None, zero_division=0)
    resumen.append({"Modelo": nombre, "F1 macro": por_clase.mean(),
                    "Exactitud": accuracy_score(reales, pred), "Kappa": cohen_kappa_score(reales, pred),
                    **dict(zip(clases_v6, por_clase))})
tabla = pd.DataFrame(resumen).set_index("Modelo")
display(tabla.round(4))

matrices = np.array([[confusion_matrix(g.etapa, g[col], labels=clases_v6) for col in modelos]
                     for _, g in resultados.groupby("subject_id", sort=True)])
rng = np.random.default_rng(42)
indices = rng.integers(len(matrices), size=(5000, len(matrices)))
cm = matrices[indices].sum(axis=1)
den = cm.sum(axis=-1) + cm.sum(axis=-2)
f1 = np.divide(2 * np.diagonal(cm, axis1=-2, axis2=-1), den,
               out=np.zeros(den.shape, dtype=float), where=den != 0).mean(axis=-1)
lo, hi = np.quantile(f1[:, 0] - f1[:, 1], [0.025, 0.975])
delta = tabla.at["DART + CRNN", "F1 macro"] - tabla.at["V6 fase C", "F1 macro"]
print(f"{len(resultados):,} segmentos · {len(prueba)} noches · {len(elegidos)} personas de Cassette")
print(f"Diferencia de F1 macro (mixto menos V6): {delta:+.4f}. IC 95 % [{lo:+.4f}, {hi:+.4f}].")
if lo <= 0 <= hi:
    print("El intervalo incluye cero. La comparación es inconclusa sobre la superioridad de un método.")
else:
    mejor = "DART + CRNN" if delta > 0 else "V6 fase C"
    print(f"En esta muestra retrospectiva de Cassette, la diferencia favorece a {mejor}.")

,F1 macro,Exactitud,Kappa,W,N1,N2,N3,REM
Modelo,,,,,,,,
DART + CRNN,0.6855,0.7947,0.7055,0.9137,0.3467,0.8013,0.5333,0.8325
V6 fase C,0.6432,0.7727,0.6717,0.9122,0.2735,0.7780,0.4594,0.7930


26,653 segmentos · 19 noches · 10 personas de Cassette
Diferencia de F1 macro (mixto menos V6): +0.0423. IC 95 % [+0.0148, +0.0645].
En esta muestra retrospectiva de Cassette, la diferencia favorece a DART + CRNN.
